# Análisis granulométrico de roca — pipeline en Colab
Ejecuta el código del repositorio (`ml/` + `scripts/`) en lugar de copiar celdas. Organizado por fases CRISP-DM.

**Entorno:** Runtime → Change runtime type → T4 GPU.

In [ ]:
# 0. Clonar el repositorio (si es privado, usar un token: https://<TOKEN>@github.com/...)
!git clone https://github.com/DaniGuevara09/analisis_rocas_ic.git
%cd analisis_rocas_ic/ia
!pip install -q -r requirements.txt

In [ ]:
# Montar Drive: dataset, Excel de escalas y pesos viven en Drive, no en Git
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/ThesisProject/Colab'
os.environ['ROCAS_DATA']    = f'{DRIVE}/data'
os.environ['ROCAS_WEIGHTS'] = f'{DRIVE}/results'   # unet_best.pth y mobile_sam.pt ya están aquí
os.environ['ROCAS_YOLO']    = f'{DRIVE}/results/yolo/yolo11m_seg_rocks-7/weights/best.pt'
os.environ['ROCAS_RESULTS'] = f'{DRIVE}/results'


## Fase 2–3 · Comprensión y preparación de los datos
Imágenes de Wilder segmentadas manualmente (LabelMe/CVAT) → COCO → train/val/test + etiquetas YOLO.

In [ ]:
# Caso LabelMe (un .json por imagen):
!python scripts/preparar_dataset.py --labelme $ROCAS_DATA/raw/wilder --images $ROCAS_DATA/raw/wilder
# Caso COCO exportado desde CVAT/Roboflow:
# !python scripts/preparar_dataset.py --coco $ROCAS_DATA/raw/wilder_coco.json --images $ROCAS_DATA/raw/wilder

In [ ]:
# Excel con N° de imagen + escala del objeto de referencia
!python scripts/crear_excel_escalas.py --images $ROCAS_DATA/dataset/images/test --out $ROCAS_DATA/escalas_imagenes.xlsx
# ... llenar el Excel (o usar scripts/marcar_escala.py en un PC) y luego verificar:
!python scripts/crear_excel_escalas.py --leer $ROCAS_DATA/escalas_imagenes.xlsx

## Fase 4 · Modelado

In [ ]:
# Solo si hay dataset nuevo. Los pesos actuales ya están en Drive.
# !python scripts/entrenar.py yolo --epochs 100 --batch 4

In [ ]:
# U-Net con mapa de pesos de borde (Ronneberger 2015). Para reutilizar los pesos viejos: --resume <ruta>/unet_best.pth
# !python scripts/entrenar.py unet --epochs 80 --batch 4

## Fase 5 · Evaluación
Métricas del documento + AUC-ROC, curva ROC, IoU, Dice, AUC-PR, Boundary IoU y error de D50/D80.

In [ ]:
!pip install -q hdbscan git+https://github.com/ChaoningZhang/MobileSAM.git timm
!python scripts/evaluar_modelos.py --modelos yolo unet hdbscan mobilesam

In [ ]:
# ¿Semántica o instancia?
!python scripts/semantica_vs_instancia.py

In [ ]:
# ¿U-Net mejora los bordes de YOLO?
!python scripts/evaluar_modelos.py --modelos yolo yolo_unet --out $ROCAS_RESULTS/bordes_unet

In [ ]:
import pandas as pd
from IPython.display import Image, display
R = os.environ['ROCAS_RESULTS']
display(pd.read_csv(f'{R}/benchmark/tabla_metricas.csv'))
display(Image(f'{R}/benchmark/roc_pixel.png', width=520))
display(Image(f'{R}/benchmark/pr_instancia.png', width=520))

## Fase 6 · Despliegue
Si un modelo nuevo mejora el benchmark, su `best.pt` se copia a `backend/modelos/best.pt`. El backend lo carga con `SegmentadorYOLO` (tarea T-007). Ver `docs/ia.md`.